# Exploração dos dados do SIH: internações em SP (jan/2023)

Primeira exploração dos dados de internações hospitalares do SUS (SIH/DATASUS, grupo RD) para o estado de São Paulo em janeiro de 2023.

**Objetivo do projeto:** prever internações por doenças respiratórias para ajudar a antecipar a pressão sobre leitos.
**Objetivo deste notebook:** entender o dado antes de modelar (estrutura, datas, qualidade e armadilhas).

**Privacidade:** os dados originais têm colunas que podem identificar pessoas (CEP, nascimento, número da AIH). Este notebook usa só as colunas necessárias e mostra apenas resultados agregados.

In [ ]:
import pysus

arquivos = pysus.ftp.sih("SP", 2023, 1, download=False)
arquivos

In [ ]:
df = pysus.ftp.sih("SP", 2023, 1, group="RD", as_dataframe=True)
df.shape

## 1. Leitura dos dados

O arquivo baixado com o PySUS tem 210.225 linhas e 113 colunas. Para economizar memória e evitar dados pessoais, leio só as colunas úteis para o projeto: datas, diagnóstico, município, idade, sexo, permanência, óbito, caráter da internação, dias de UTI (`UTI_INT_TO`), valor total e o tipo de AIH (`IDENT`, `SEQ_AIH5`).

Observação: `UTI_MES_TO` indica quantidade de atos de UTI, não dias. Para dias de UTI, o layout oficial indica `UTI_INT_TO`.

In [ ]:
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.to_parquet("../data/sih_sp_2023_01.parquet")

In [ ]:
import pandas as pd

cols = ["ANO_CMPT", "MES_CMPT", "MUNIC_RES", "MUNIC_MOV", "DT_INTER",
        "DT_SAIDA", "DIAG_PRINC", "SEXO", "COD_IDADE", "IDADE",
        "DIAS_PERM", "MORTE", "CAR_INT", "UTI_INT_TO", "VAL_TOT",
        "IDENT", "SEQ_AIH5"]

# lê só as 15 colunas que importam, direto do arquivo salvo
sih = pd.read_parquet("../data/sih_sp_2023_01.parquet", columns=cols)
sih.shape

## 2. Conversão das datas

As datas vêm como texto no formato AAAAMMDD. Converto `DT_INTER` (entrada) e `DT_SAIDA` (saída) para data de verdade e olho o intervalo.

**Resultado:** as entradas vão de 01/01/2008 a 31/01/2023. Uma data de 2008 num arquivo de 2023 chamou atenção e foi investigada nas próximas seções.

In [ ]:
sih["DT_INTER"] = pd.to_datetime(sih["DT_INTER"], format="%Y%m%d", errors="coerce")
sih["DT_SAIDA"] = pd.to_datetime(sih["DT_SAIDA"], format="%Y%m%d", errors="coerce")
sih["DT_INTER"].min(), sih["DT_INTER"].max()

## 3. Internações respiratórias

O diagnóstico principal (`DIAG_PRINC`) vem em CID-10. As doenças respiratórias são o capítulo J (códigos que começam com "J").

**Resultado:** 16.281 internações respiratórias, cerca de 7,7% do total do mês (1 a cada 13).
**Hipótese (ainda não testada):** em janeiro (verão) a proporção é baixa e deve subir no outono e no inverno. Isso será conferido com mais meses.

In [ ]:
resp = sih[sih["DIAG_PRINC"].str.startswith("J", na=False)]
len(resp), len(resp) / len(sih)

## 4. Entradas anteriores a 2023

**Resultado:** 71.616 internações (34% do arquivo) têm data de entrada anterior a 2023. A distribuição por mês de entrada forma uma "cauda": 138.609 em jan/2023, 46.221 em dez/2022, 14.269 em nov/2022, e cai gradualmente até ago/2022.

**Interpretação:** o arquivo reúne internações processadas em janeiro (competência `ANO_CMPT`/`MES_CMPT`), e não apenas as que entraram em janeiro. Quem entrou em dezembro e teve alta em janeiro aparece aqui, e hospitais com atraso de cobrança também.

In [ ]:
antes = sih[sih["DT_INTER"] < "2023-01-01"]
len(antes), len(antes) / len(sih)

## 5. Datas de saída

**Resultado:** 162.351 internações (77%) tiveram saída em jan/2023, e as outras 23% saíram antes (31.154 em dez/2022, 10.842 em nov/2022, 5.091 em out/2022). Então o arquivo não é organizado nem pela data de entrada nem pela de saída: reflete o mês em que a cobrança foi processada.

In [ ]:
sih["DT_INTER"].dt.to_period("M").value_counts().sort_index().tail(6)

In [ ]:
antes["DIAS_PERM"].describe()

In [ ]:
sih["DT_SAIDA"].dt.to_period("M").value_counts().sort_index().tail(4)

## 6. Tipos das colunas

Algumas colunas numéricas (`DIAS_PERM`, `IDADE`, `VAL_TOT`, `UTI_INT_TO`) vinham como texto e foram convertidas para número, para permitir cálculos. As colunas de código (sexo, óbito, caráter da internação, município) continuam como texto, que é o formato correto para códigos.

In [ ]:
for c in ["DIAS_PERM", "IDADE", "VAL_TOT", "UTI_INT_TO"]:
    sih[c] = pd.to_numeric(sih[c], errors="coerce")

sih.dtypes

## 7. Duração das internações

Calculei a duração pela diferença entre saída e entrada e comparei com o `DIAS_PERM` registrado pelo sistema.

**Resultado:** mediana de 3 dias, 75% até 6 dias. A média (46 dias) é puxada por poucos casos extremos, e o máximo chega a 5.509 dias (cerca de 15 anos). Há 2.505 registros (1,2%) com mais de 365 dias entre entrada e saída, mas o `DIAS_PERM` deles é de ~31 dias, o que indica que o registro cobre só o mês, e não a internação inteira.

In [ ]:
antes = sih[sih["DT_INTER"] < "2023-01-01"]
antes["DIAS_PERM"].describe()

In [ ]:
(sih["DT_INTER"] < "2022-07-01").sum()

In [ ]:
dias_calc = (sih["DT_SAIDA"] - sih["DT_INTER"]).dt.days
dias_calc.describe()

In [ ]:
(dias_calc > 365).sum()

In [ ]:
longas = sih[dias_calc > 365]
longas["DIAS_PERM"].describe()

## 8. Teste: são internações de longa permanência?

**Hipótese:** os registros com mais de 365 dias são AIH de longa permanência, cobradas mês a mês, e a data de entrada é a original da internação.

**Resultado:** todas as 2.505 têm `IDENT` = 5. No arquivo inteiro só existem os tipos 1 (205.499) e 5 (4.726). O layout oficial descreve a "AIH tipo 5" como longa permanência. O campo `SEQ_AIH5` veio zerado para todas, então não ajuda.

**Decisão:** na série temporal por data de entrada, excluir os registros com `IDENT` = 5 (2,2% do total).
**A confirmar:** o significado exato do `IDENT` = 1 no dicionário de dados.

In [ ]:
dias_calc = (sih["DT_SAIDA"] - sih["DT_INTER"]).dt.days
longas = sih[dias_calc > 365]

print("IDENT das longas:")
print(longas["IDENT"].value_counts(), "\n")
print("IDENT de todas:")
print(sih["IDENT"].value_counts(), "\n")
print("SEQ_AIH5 das longas (top 5):")
print(longas["SEQ_AIH5"].value_counts().head())

## Conclusões e próximos passos

**O que aprendi sobre os dados**
- O arquivo mensal reúne internações por data de processamento, não de entrada.
- 34% dos registros de jan/2023 entraram antes de 2023.
- Registros de longa permanência (`IDENT` = 5) distorcem a data de entrada e devem ser excluídos da série.
- Cerca de 7,7% das internações do mês são respiratórias.

**Decisões pendentes**
- Descartar os últimos 2 a 3 meses do período baixado, que ainda estarão incompletos.
- Tratar o período da pandemia (2020-2021), que distorce a sazonalidade.
- Confirmar no dicionário oficial os códigos de `COD_IDADE`, `CAR_INT` e `IDENT`.

**Próximos passos**
1. Ler o dicionário de dados e documentar os códigos.
2. Baixar vários meses e consolidar a base.
3. Montar a série mensal de internações respiratórias.